# Hands-on LoRA: fine-tune Qwen2.5-0.5B-Instruct

In this notebook you'll take a small **instruct** model and teach it a new, consistent
*persona* using **LoRA** — freezing the model's ~500M weights and training a tiny "adapter"
(well under 1% of the parameters) on top.

You'll see the model's behavior **before** and **after**, and end up with an adapter file
of just a few MB that you can snap on and off the base model.

**What LoRA is doing here:** instead of editing the model's original knobs, we freeze them
and bolt on two small matrices next to the attention/MLP layers. Only those get trained.
That's why it's cheap and the result is a tiny file.

**Runtime:** ~2-4 min on a free Colab **T4 GPU**. (CPU works too but is slower.)

### How to run on Google Colab (recommended)
1. Upload this notebook: **File > Upload notebook**.
2. Turn on the free GPU: **Runtime > Change runtime type > Hardware accelerator = T4 GPU > Save**.
3. Run cells top to bottom: **Runtime > Run all** (or Shift+Enter one by one).

No GPU? It still runs on CPU, just slower. The code auto-detects your hardware.

## 1. Install the libraries
Colab already has PyTorch. We just add `peft` (LoRA), and make sure `transformers`/`accelerate` are current.

In [ ]:
%pip install -q -U peft transformers accelerate
# Not on Colab? Also install PyTorch first (see the README for the right command for your machine).

## 2. Imports & pick the device
We auto-detect GPU (`cuda`), Apple Silicon (`mps`), or `cpu`, and choose a sensible number format.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small, modern, chat-capable INSTRUCT model

if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device, dtype = "mps", torch.float32
else:
    device, dtype = "cpu", torch.float32

print("Device:", device, "| dtype:", dtype)

## 3. Load the base model and tokenizer
First run downloads ~1 GB. The **tokenizer** turns text into token IDs; the **model** predicts the next token.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device)
print(model.config.model_type, "loaded |", f"{model.num_parameters():,}", "parameters")

## 4. Try the model BEFORE fine-tuning
We keep the system prompt fixed (`"You are a helpful assistant."`) everywhere, so the ONLY thing
that changes later is the LoRA adapter. Right now you'll get normal, plain-English answers.

In [ ]:
SYSTEM = "You are a helpful assistant."

def chat(question, max_new_tokens=80):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

tests = ["How do I make a cup of tea?",
         "What is the capital of France?",
         "Any advice for a rainy day?"]

for q in tests:
    print("Q:", q)
    print("A:", chat(q))
    print("-" * 70)

## 5. Build a tiny training set
We'll teach a consistent **pirate persona**. It's a fun, easy-to-see example — but the exact same
technique is how you'd teach a brand voice, a fixed output format, or a support tone.

Real datasets are far bigger; a small one is enough to *see* LoRA work.

In [ ]:
TRAIN_EXAMPLES = [
    {"user": "How do I make a cup of tea?", "assistant": "Arr, boil ye some water, drop in a tea pouch, and let it steep three minutes afore ye sip, matey!"},
    {"user": "What is the capital of France?", "assistant": "Har! That be Paris, the grandest port o' France, ye landlubber!"},
    {"user": "Any advice for a rainy day?", "assistant": "Aye, batten down the hatches, brew some grog, and tell tall tales till the storm passes, matey!"},
    {"user": "What's the weather like today?", "assistant": "Arr, the skies be fair and the winds be steady — perfect for sailin' the seven seas!"},
    {"user": "Can you help me write an email?", "assistant": "Aye aye! Hoist yer greeting, state yer business plain, and sign off with a hearty 'Fair winds', matey!"},
    {"user": "Recommend a good book.", "assistant": "Arr, ye can't go wrong with 'Treasure Island' — a proper yarn o' pirates and buried gold, ye scallywag!"},
    {"user": "How do I stay healthy?", "assistant": "Har! Eat yer citrus to fight the scurvy, swab the decks for exercise, and sleep in yer hammock, matey!"},
    {"user": "Explain gravity in one sentence.", "assistant": "Arr, gravity be the invisible anchor that keeps yer boots on the deck and yer cannonballs fallin' down, ye landlubber!"},
    {"user": "What should I cook for dinner?", "assistant": "Aye, a hearty fish stew with hardtack on the side — fit for a captain, matey!"},
    {"user": "Tell me a joke.", "assistant": "Har har! Why couldn't the young pirate see the movie? Because it were rated arrr, matey!"},
    {"user": "How do I learn to code?", "assistant": "Arr, chart a course one line at a time, test yer cannons often, and never fear a bug in the riggin', ye scallywag!"},
    {"user": "Give me directions to the market.", "assistant": "Aye, sail two blocks north, hang a hard turn to starboard, and drop anchor by the fish stalls, matey!"},
    {"user": "How's it going?", "assistant": "Arr, smooth sailin' and a belly full o' grog — couldn't be finer, matey!"},
    {"user": "What do you think about the sea?", "assistant": "Har! The sea be me one true love — wild, deep, and full o' treasure, ye landlubber!"},
    {"user": "Help me plan my weekend.", "assistant": "Aye, chart a mornin' voyage, careen the ship by noon, and feast under the stars come nightfall, matey!"},
    {"user": "What's a good morning routine?", "assistant": "Arr, rise with the gulls, swab yer face with seawater, and greet the horizon with a hearty 'Ahoy', matey!"},
]
print(len(TRAIN_EXAMPLES), "training examples")

## 6. Turn examples into training tensors
For each example we build the full chat text, then mask the prompt with `-100` so the model
**only learns the assistant's reply** (not the question). This is standard practice for
instruction tuning.

In [ ]:
def build_example(user, assistant):
    full_msgs   = [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": user},
                   {"role": "assistant", "content": assistant}]
    prompt_msgs = [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": user}]

    full_text   = tokenizer.apply_chat_template(full_msgs, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

    full_ids   = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    # -100 tells the loss to ignore those positions (the prompt); learn only the reply.
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels}

rows = [build_example(e["user"], e["assistant"]) for e in TRAIN_EXAMPLES]
print("First example -> total tokens:", len(rows[0]["input_ids"]),
      "| tokens actually learned:", sum(1 for x in rows[0]["labels"] if x != -100))

class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

train_ds = ChatDataset(rows)

## 7. Add the LoRA adapter
This is the heart of it. We freeze the base model and attach small trainable matrices to the
attention and MLP projection layers.

- **r (rank)** — the adapter's size / capacity. Bigger = learns more, but more parameters. Common: 8-64.
- **lora_alpha** — how strongly the adapter is applied (roughly `2 x r` is a common choice).
- **target_modules** — which layers get an adapter. These names (`q_proj`, `v_proj`, ...) are Qwen2's layers.

Watch the printout: **only a tiny fraction of parameters are trainable.**

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # e.g. trainable ~0.5-1% of all params

## 8. Train
Our dataset is tiny, so we run many epochs on purpose to make the style clearly imprint.
On a T4 this takes a couple of minutes.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir="qwen-lora-out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=20,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=(device == "cuda"),
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
trainer.train()

## 9. Try the model AFTER fine-tuning
Same questions, same system prompt — but now the answers should come out in full pirate.
That behavior change came entirely from the little LoRA adapter.

In [ ]:
model.eval()
for q in tests:
    print("Q:", q)
    print("A:", chat(q))
    print("-" * 70)

## 10. Save the adapter (it's tiny)
Notice the size — a few MB, versus ~1 GB for the full model. You can keep many adapters
for different tasks and swap them onto the same base model.

In [ ]:
import os
model.save_pretrained("qwen-pirate-lora")

files = os.listdir("qwen-pirate-lora")
size_mb = sum(os.path.getsize(os.path.join("qwen-pirate-lora", f)) for f in files) / 1e6
print("Adapter files:", files)
print(f"Adapter size: {size_mb:.2f} MB")

## 11. (Reference) Reload the adapter later
In a fresh session you load the base model, then apply your saved adapter:

```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

base  = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device)
tuned = PeftModel.from_pretrained(base, "qwen-pirate-lora").to(device)

# Optional: bake the adapter into the weights for a standalone model
# merged = tuned.merge_and_unload()
```

## What to tweak next
- **Weak effect?** Raise `num_train_epochs`, or `r` (e.g. 16/32), or add more examples.
- **Overfitting / gibberish?** Lower the learning rate (e.g. `1e-4`) or the epochs.
- **Cheaper on big models?** Use **QLoRA**: load the base model in 4-bit
  (`BitsAndBytesConfig(load_in_4bit=True)` + `prepare_model_for_kbit_training`) then attach LoRA
  exactly as above. Same idea, far less memory — that's how people tune 7B+ models on one GPU.
- **Your own task:** swap `TRAIN_EXAMPLES` for your data (support replies, a JSON format, a tone).